# Segment-Level Multimodal RAG for Audiovisual Reasoning


This notebook implements a segment-level multimodal retrieval and reasoning framework for long-form audio-visual datasets. The system preprocesses videos and audio, segments them temporally, embeds them in a shared multimodal space using ImageBind, and performs Retrieval-Augmented Generation (RAG) followed by multimodal reasoning with Qwen2.5-Omni.

The goal is to retrieve temporally precise audio-visual segments ("needles") from long multi-video corpora ("haystacks") and reason over them to answer structured questions.


## Pipeline Overview

### 1. Dataset Preparation

* Dataset integrity checks (video, audio, captions)
* Media renaming and filtering of QA files
* Standardization of video (H.264) and audio (PCM WAV)

### 2. Temporal Segmentation

* Deterministic video and audio segmentation
* Optional remainder merging for stable segment boundaries
* Caption segmentation aligned with media timestamps
* Configurable segment lengths (e.g., 30s, 60s, 200s)

### 3. Multimodal Embedding (ImageBind)

* Segment-level embedding extraction
* Shared embedding space for text, audio, and vision
* Cosine similarity retrieval
* Support for:

  * Local (per-topic) embeddings
  * Global embeddings across topics
* Cached `.pt` embedding loading for scalable experimentation

### 4. Audio-Visual Retrieval (AV-RAG)

* Text query → Top-K segment retrieval
* Configurable Top-K selection
* Local vs global retrieval modes
* Designed for temporal and multi-hop reasoning over segmented media

### 5. Multimodal Reasoning (Qwen2.5-Omni)

* Retrieved segments passed to Qwen Omni
* Structured system prompt with:

  * Relevance filtering (Unanswerable detection)
  * Timestamp-grounded answers
* Batch inference with GPU-aware execution


## Features

* Multi-topic dataset support (Job Interviews, Medical Consultations, Customer Service, etc.)
* Deterministic segmentation for audio-visual alignment
* Shared multimodal embedding retrieval via ImageBind
* Local vs global retrieval bank comparison
* Temporal RAG over long-form video
* Integration with Qwen2.5-Omni for multimodal reasoning
* Scalable embedding caching and reproducible configuration blocks
* Designed for multi-hop and cross-segment reasoning experiments

## Dataset

This project uses a filtered subset of the **SONIC-O1** benchmark
([https://huggingface.co/datasets/vector-institute/sonic-o1](https://huggingface.co/datasets/vector-institute/sonic-o1)).

SONIC-O1 is a real-world multimodal benchmark for evaluating joint **audio–video understanding**. It contains 231 long-form conversational videos (~60 hours total) with synchronized video, audio, captions, and human-verified annotations across multiple tasks.

### Subset Used

We focus on **Task 2: Multiple-Choice QA (MCQ)** and use both the VQA annotations and corresponding media.

To ensure tractable segment-level retrieval, we restrict the dataset to:

* **Three domains**

  * Patient–Doctor Consultations
  * Job Interviews
  * Customer Service Interactions

* **Videos ≤ 120 seconds**

This filtered subset is used for segment-level multimodal retrieval and reasoning experiments (Audio-Visual RAG + Qwen2.5-Omni).

**License:** Vector Institute License.
Attribution: “Built with Vector Institute SONIC-O1.”


## Preprocessing data folder

Each folder within data refers to a topic of videos which would contain a audio/ video/ folder each along with a <topic>.json that contains the questions 

In [20]:
import json # To read and write JSON files
import os 
from src.media_utils import list_video_durations # To get the duration of each video in a folder
from src.dataset_utils import (
    check_dataset_integrity, # To check the integrity of the dataset (e.g., if all videos are present and not corrupted)
    rename_media_files, # To rename media files in a consistent format (e.g., video_001.mp4, video_002.mp4, etc.) for easier processing and organization
    filter_json_by_existing_videos, # To filter the QA JSON file based on the existing videos in the dataset
    simplify_mcq_json # To simplify the MCQ JSON file by removing unnecessary fields and keeping only the relevant information for training
)

In [ ]:
list_video_durations("./data/Customer_Service_Interactions/video",
                     120)

In [ ]:
check_dataset_integrity("./data",
                       vid_dir="video",
                           aud_dir="audio",
                           cap_dir="caption")

In [ ]:
# Rename media files in a consistent format for easier processing and organization
topic = "Customer_Service_Interactions" # "Patient-Doctor_Consultations" # "Job_Interviews" # "Customer_Service_Interactions"
rename_media_files(f"./data/{topic}/")

In [ ]:
# Filter the QA JSON file based on the existing videos in the dataset
topic = "Customer_Service_Interactions" # "Patient-Doctor_Consultations" # "Job_Interviews" # "Customer_Service_Interactions"
filter_json_by_existing_videos(
    json_path=f"./data/{topic}.json",
    video_folder=f"./data/{topic}/video",
    output_path=f"./data/{topic}_filtered.json"
)

# Simplify the MCQ JSON file by removing unnecessary fields and keeping only the relevant information for training
with open(f"./data/{topic}_filtered.json", "r") as f:
    data = json.load(f)

new_data = simplify_mcq_json(data)

with open(f"./data/{topic}_filtered.json", "w") as f:
    json.dump(new_data, f, indent=2)

## Standardize Audios and Videos
The audios and videos are to be converted into a standardized format (H.264 for video and AAC for audio) while preserving the original quality as much as possible.

In [69]:
from src.media_utils import (
    process_video,
    process_audio,
)

In [70]:
def process_media(topic, max_duration=300):
    """
    Process video and audio files for a given topic.

    Parameters:
    - topic (str): The topic name (e.g., "Customer_Service_Interactions").
    - max_duration (int): Maximum duration for processing (default: 300 seconds).
    """
    VIDEO_DIR = f"./data/{topic}/video"
    AUDIO_DIR = f"./data/{topic}/audio"
    if max_duration is not None:
        VIDEO_PROCESS_DIR = f"./data/{topic}/process-video_{max_duration}s"
        AUDIO_PROCESS_DIR = f"./data/{topic}/process-audio_{max_duration}s"
    else:
        VIDEO_PROCESS_DIR = f"./data/{topic}/process-video"
        AUDIO_PROCESS_DIR = f"./data/{topic}/process-audio"

    # Run processing
    if VIDEO_DIR is not None:
        process_video(VIDEO_DIR, VIDEO_PROCESS_DIR, max_duration)

    if AUDIO_DIR is not None:
        process_audio(AUDIO_DIR, AUDIO_PROCESS_DIR, max_duration)

In [ ]:
# Example usage
topic = "Customer_Service_Interactions" # "Patient-Doctor_Consultations" # "Job_Interviews" # "Customer_Service_Interactions"
process_media(topic=topic, max_duration=None)

## Split Audio and Video into segments for processing


In [86]:
from src.segmentation_utils import (
    save_segmented_srt,
    split_audio,
    split_video,
)
from src.dataset_utils import (
    parse_srt_with_timestamps,

)

In [87]:
def process_and_split_media(
    video_dir,
    audio_dir,
    caption_dir,
    segment_video_dir,
    segment_audio_dir,
    segment_caption_dir,
    segment_length=30,
    max_files=None
):
    """
    Process and split video, audio, and captions into smaller segments.
    """

    os.makedirs(segment_video_dir, exist_ok=True)
    os.makedirs(segment_audio_dir, exist_ok=True)
    os.makedirs(segment_caption_dir, exist_ok=True)

    # ---- Split video ----
    if os.path.exists(video_dir):
        split_video(video_dir, segment_video_dir, segment_length, max_files=max_files)

    # ---- Split audio ----
    if os.path.exists(audio_dir):
        split_audio(audio_dir, segment_audio_dir, segment_length, max_files=max_files)

    # ---- Split captions ----
    if not os.path.exists(caption_dir):
        return

    segment_video_files = os.listdir(segment_video_dir)

    for filename in sorted(os.listdir(caption_dir)):

        if not filename.endswith(".srt"):
            continue

        video_id = os.path.splitext(filename)[0]

        total_segments = sum(
            1 for f in segment_video_files
            if f.startswith(video_id + "__") and f.endswith(".mp4")
        )

        print(f"{video_id} -> {total_segments} segments")

        if total_segments == 0:
            continue

        srt_path = os.path.join(caption_dir, filename)
        entries = parse_srt_with_timestamps(srt_path)

        save_segmented_srt(
            entries=entries,
            segment_length=segment_length,
            video_id=video_id,
            output_dir=segment_caption_dir,
            total_segments=total_segments
        )

In [ ]:
topic = "Patient-Doctor_Consultations" # "Patient-Doctor_Consultations" # "Job_Interviews" # "Customer_Service_Interactions"
base_dir = f"./data/{topic}"
segment_length = 30
video_dir = os.path.join(base_dir, "process-video")
audio_dir = os.path.join(base_dir, "process-audio")
caption_dir = os.path.join(base_dir, "caption")

segment_video_dir = os.path.join(base_dir, f"segment-video_{segment_length}s")
segment_audio_dir = os.path.join(base_dir, f"segment-audio_{segment_length}s")
segment_caption_dir = os.path.join(base_dir, f"segment-caption_{segment_length}s")

process_and_split_media(
    video_dir=video_dir,
    audio_dir=audio_dir,
    caption_dir=caption_dir,
    segment_video_dir=segment_video_dir,
    segment_audio_dir=segment_audio_dir,
    segment_caption_dir=segment_caption_dir,
    segment_length=segment_length,
    # max_files=4
)

In [101]:
check_dataset_integrity("./data",
                       vid_dir="segment-video_30s",
                           aud_dir="segment-audio_30s",
                           cap_dir="segment-caption_30s")


Checking: Customer_Service_Interactions
  Total unique IDs: 19
  Matched triplets: 19
  Mismatches: 0
  All video-audio-caption triplets match.

Checking: Job_Interviews
  Total unique IDs: 15
  Matched triplets: 15
  Mismatches: 0
  All video-audio-caption triplets match.

Checking: Patient-Doctor_Consultations
  Total unique IDs: 15
  Matched triplets: 15
  Mismatches: 0
  All video-audio-caption triplets match.

Checking: misc
  Total unique IDs: 0
  Matched triplets: 0
  Mismatches: 0
  All video-audio-caption triplets match.


## Audio-Visual Retrieval-Augmented Generation (AV-RAG)

In [ ]:
import sys
# sys.path.append("./src/ImageBind")
import os
import json
import argparse
from src.model.avrag import AVRAG
from imagebind.models.imagebind_model import ModalityType

In [2]:
def run_avrag(
    model_path: str,
    annotations: str,
    output_path: str,
    video_vocabs: str,
    audio_vocabs: str,
    caption_vocabs: str,
    batch_size: int,
    use_cache: bool,
    topk: int,
    max_questions: int = None
):
    """
    Run AVRAG retrieval on the given dataset.

    Args:
    - model_path (str): Path to the AVRAG model checkpoint.
    - annotations (str): Path to the JSON file containing questions.
    - output_path (str): Path to save the retrieval results.
    - video_vocabs (str): Path to video files or precomputed video embeddings.
    - audio_vocabs (str): Path to audio files or precomputed audio embeddings.
    - caption_vocabs (str): Path to caption files or precomputed caption embeddings.
    - batch_size (int): Batch size for encoding.
    - use_cache (bool): Whether to use precomputed embeddings.
    - topk (int): Number of top results to retrieve.
    - max_questions (int, optional): Maximum number of questions to process.
    """
    # ===== LOAD QUESTIONS =====
    with open(annotations, "r") as f:
        sources = json.load(f)

    if max_questions is not None:
        sources = sources[:max_questions]

    # ===== INITIALIZE RAG =====
    rag = AVRAG(model_path=model_path, bsz=batch_size)

    # ===== PREPARE MEDIA PATHS =====
    if use_cache:
        video_paths = os.path.join(video_vocabs, "video_embeddings.pt")
        audio_paths = os.path.join(audio_vocabs, "audio_embeddings.pt")
        caption_paths = os.path.join(caption_vocabs, "caption_embeddings.pt")
    else:
        video_paths = video_vocabs
        audio_paths = audio_vocabs
        caption_paths = caption_vocabs

    # ===== ENCODE MEDIA =====
    print("Encoding media...")

    v_embed = rag.encode(video_paths, ModalityType.VISION, cache=use_cache)
    a_embed = rag.encode(audio_paths, ModalityType.AUDIO, cache=use_cache)
    c_embed = rag.encode_srt_dir(caption_paths, cache=use_cache)

    # ===== RETRIEVAL =====
    print("Running retrieval...")
    targets = []

    for idx, source in enumerate(sources):

        question = source["question"]

        # Encode text
        t_embed = rag.encode([question], ModalityType.TEXT)

        # Joint AV + caption retrieval
        res = rag.joint_rag(
            query=t_embed,
            vocab_vision=v_embed,
            vocab_audio=a_embed,
            vocab_caption=c_embed,
            k=topk
        )

        # Extract clean file IDs
        retrieved_ids = [
            item["file"] for item in res[0][t_embed["filename"][0]]
        ]

        source["retrieved_file"] = retrieved_ids
        targets.append(source)

        print(f"[{idx+1}/{len(sources)}] Retrieved: {retrieved_ids}")

    # ===== SAVE OUTPUT =====
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, "w") as f:
        json.dump(targets, f, indent=2)

    print("Finished retrieval. Saved to:", output_path)

In [14]:
# Define the topic for processing
topic = "Customer_Service_Interactions"  # "Patient-Doctor_Consultations" # "Job_Interviews" # "Customer_Service_Interactions"
term = ""
suffix = "30s"

# Path to the pre-trained model checkpoint
MODEL_PATH = "./checkpoints/imagebind_huge.pth"

# Path to the annotations file for the selected topic
ANNOTATIONS = f"./data/{topic}_filtered.json"

# Path to save the output of the retrieval process
OUTPUT_PATH = f"./output/{topic}_retrieved_{term}_{suffix}.json"

# Directories containing segmented video, audio, and caption files for the selected topic
VIDEO_VOCABS = f"./data/{topic}/segment-video_{suffix}"
AUDIO_VOCABS = f"./data/{topic}/segment-audio_{suffix}"
CAPTION_VOCABS = f"./data/{topic}/segment-caption_{suffix}"

# Batch size for processing
BATCH_SIZE = 8

# Whether to use cached embeddings (set to True to use cache, False to recompute)
USE_CACHE = False

# Number of top-k results to retrieve
TOPK = 6

In [ ]:
# Run the AVRAG (Audio-Visual Retrieval and Generation) process
run_avrag(
    model_path=MODEL_PATH,       # Path to the pre-trained model checkpoint
    annotations=ANNOTATIONS,    # Path to the annotations file for the selected topic
    output_path=OUTPUT_PATH,    # Path to save the output of the retrieval process
    video_vocabs=VIDEO_VOCABS,  # Directory containing segmented video files
    audio_vocabs=AUDIO_VOCABS,  # Directory containing segmented audio files
    caption_vocabs=CAPTION_VOCABS, # Directory containing segmented caption files
    batch_size=BATCH_SIZE,      # Batch size for processing
    use_cache=USE_CACHE,        # Whether to use cached embeddings or recompute them
    topk=TOPK,                  # Number of top-k results to retrieve
    # max_questions=8,           # Maximum number of questions to process
)

### Build gloval embeddings across topics

In [3]:
import os
import torch
import shutil

def collect_files_from_dir(directory, extensions):
    """
    Collect files with specific extensions from a directory.

    Parameters:
    - directory (str): Path to the directory to search.
    - extensions (tuple): File extensions to include (e.g., (".mp4", ".wav")).

    Returns:
    - list: List of file paths matching the specified extensions.
    """
    files = []
    for f in os.listdir(directory):
        if f.startswith("."):  # Skip hidden files
            continue
        full_path = os.path.join(directory, f)
        if os.path.isfile(full_path) and f.endswith(extensions):
            files.append(full_path)
    return files


def build_global_embeddings(
    root_data_dir,
    model_path,
    segment_suffix="30s",
    output_dir="./data/global_embeddings"
):
    """
    Build global embeddings for video, audio, and captions across multiple topics.

    Parameters:
    - root_data_dir (str): Root directory containing topic subdirectories.
    - model_path (str): Path to the pre-trained model checkpoint.
    - segment_suffix (str): Suffix for segmented media directories (default: "30s").
    - output_dir (str): Directory to save the global embeddings (default: "./data/global_embeddings").
    """
    os.makedirs(output_dir, exist_ok=True)

    # Initialize the AVRAG model
    rag = AVRAG(model_path=model_path, bsz=8)

    all_video_files = []  # List to store all video file paths
    all_audio_files = []  # List to store all audio file paths

    # Temporary directory to store all caption files
    temp_caption_dir = os.path.join(output_dir, "__all_captions_tmp")
    os.makedirs(temp_caption_dir, exist_ok=True)

    # Iterate through each topic in the root data directory
    for topic in os.listdir(root_data_dir):
        topic_path = os.path.join(root_data_dir, topic)

        # Skip non-directories and hidden topics
        if not os.path.isdir(topic_path) or topic.startswith("."):
            continue

        # Define paths for segmented media directories
        video_dir = os.path.join(topic_path, f"segment-video_{segment_suffix}")
        audio_dir = os.path.join(topic_path, f"segment-audio_{segment_suffix}")
        caption_dir = os.path.join(topic_path, f"segment-caption_{segment_suffix}")

        # Collect and copy video files
        if os.path.exists(video_dir):
            for f in os.listdir(video_dir):
                if f.endswith(".mp4"):
                    src = os.path.join(video_dir, f)
                    dst = os.path.join(output_dir, f"{topic}__{f}")
                    shutil.copy(src, dst)
                    all_video_files.append(dst)

        # Collect and copy audio files
        if os.path.exists(audio_dir):
            for f in os.listdir(audio_dir):
                if f.endswith((".wav", ".m4a")):
                    src = os.path.join(audio_dir, f)
                    dst = os.path.join(output_dir, f"{topic}__{f}")
                    shutil.copy(src, dst)
                    all_audio_files.append(dst)

        # Collect and copy caption files
        if os.path.exists(caption_dir):
            for f in os.listdir(caption_dir):
                if f.endswith(".srt"):
                    src = os.path.join(caption_dir, f)
                    dst = os.path.join(temp_caption_dir, f"{topic}__{f}")
                    shutil.copy(src, dst)

    # Encode video files
    print("Encoding videos...")
    video_embed = rag.encode(all_video_files, ModalityType.VISION, cache=False)

    # Encode audio files
    print("Encoding audio...")
    audio_embed = rag.encode(all_audio_files, ModalityType.AUDIO, cache=False)

    # Encode caption files
    print("Encoding captions...")
    caption_embed = rag.encode_srt_dir(temp_caption_dir, cache=False)

    # Save embeddings to the output directory
    torch.save(video_embed, os.path.join(output_dir, "video_embeddings.pt"))
    torch.save(audio_embed, os.path.join(output_dir, "audio_embeddings.pt"))
    torch.save(caption_embed, os.path.join(output_dir, "caption_embeddings.pt"))

    # Remove the temporary caption directory
    shutil.rmtree(temp_caption_dir)

    print("Global embeddings saved to:", output_dir)

In [37]:
# Path to the pre-trained model checkpoint
MODEL_PATH = "./checkpoints/imagebind_huge.pth"

# Build global embeddings for video, audio, and captions across all topics
build_global_embeddings(
    root_data_dir="./data",          # Root directory containing topic subdirectories
    model_path=MODEL_PATH,           # Path to the pre-trained model checkpoint
    segment_suffix="30s",            # Suffix for segmented media directories
    output_dir="./data/global_embeddings"  # Directory to save the global embeddings
)

Encoding videos...
Encoding audio...
Encoding captions...
Global embeddings saved to: ./data/global_embeddings


### To use global embeddings

In [ ]:
# ===== CONFIG =====

# Common Configuration
topic = "Customer_Service_Interactions" # "Patient-Doctor_Consultations" # "Job_Interviews" # "Customer_Service_Interactions"
term = "global"
suffix = "30s"
BATCH_SIZE = 8
TOPK = 6
USE_CACHE = True  # IMPORTANT: now loading .pt files

# Paths
MODEL_PATH = "./checkpoints/imagebind_huge.pth"
ANNOTATIONS = f"./data/{topic}_filtered.json"
OUTPUT_PATH = f"./output/{topic}_retrieved_{term}_{suffix}.json"

# Global Embeddings
GLOBAL_VOCABS = "./data/global_embeddings"
VIDEO_VOCABS = GLOBAL_VOCABS
AUDIO_VOCABS = GLOBAL_VOCABS
CAPTION_VOCABS = GLOBAL_VOCABS

# Run AVRAG
run_avrag(
    model_path=MODEL_PATH,
    annotations=ANNOTATIONS,
    output_path=OUTPUT_PATH,
    video_vocabs=VIDEO_VOCABS,
    audio_vocabs=AUDIO_VOCABS,
    caption_vocabs=CAPTION_VOCABS,
    batch_size=BATCH_SIZE,
    use_cache=USE_CACHE, 
    topk=TOPK,
)

### To use local embeddings

In [5]:
# ===== CONFIG =====
topic = "Customer_Service_Interactions"  # "Patient-Doctor_Consultations" # "Job_Interviews" # "Customer_Service_Interactions"
term = "local"
suffix = "30s"

BATCH_SIZE = 8
USE_CACHE = True        # IMPORTANT: now loading .pt files
TOPK = 6
# ALPHA_V = 1.0

MODEL_PATH = "./checkpoints/imagebind_huge.pth"

ANNOTATIONS = f"./data/{topic}_filtered.json"

OUTPUT_PATH = f"./output/{topic}_retrieved_{term}_{suffix}.json"

# ---- LOCAL EMBEDDINGS ----
# If there is no cache
# VIDEO_VOCABS = f"./data/{topic}/segment-video_{suffix}"
# AUDIO_VOCABS = f"./data/{topic}/segment-audio_{suffix}"
# CAPTION_VOCABS = f"./data/{topic}/segment-caption_{suffix}"

# If there is cache
LOCAL_VOCABS = f"./data/{topic}/"
VIDEO_VOCABS = LOCAL_VOCABS
AUDIO_VOCABS = LOCAL_VOCABS
CAPTION_VOCABS = LOCAL_VOCABS


run_avrag(
    model_path=MODEL_PATH,
    annotations=ANNOTATIONS,
    output_path=OUTPUT_PATH,
    video_vocabs=VIDEO_VOCABS,
    audio_vocabs=AUDIO_VOCABS,
    caption_vocabs=CAPTION_VOCABS,
    batch_size=BATCH_SIZE,
    use_cache=USE_CACHE,     # must be True
    topk=TOPK,
)

Encoding media...
Running retrieval...
[1/7] Retrieved: ['002__000', '012__001', '011__000', '012__000', '008__001', '003__001']
[2/7] Retrieved: ['012__001', '002__000', '003__000', '003__001', '017__000', '017__002']
[3/7] Retrieved: ['010__002', '012__001', '003__000', '002__000', '015__000', '011__001']
[4/7] Retrieved: ['010__000', '010__002', '003__000', '012__001', '003__001', '002__000']
[5/7] Retrieved: ['012__001', '010__002', '008__001', '011__001', '017__000', '011__000']
[6/7] Retrieved: ['012__000', '012__001', '003__001', '010__001', '003__000', '017__000']
[7/7] Retrieved: ['008__001', '011__001', '003__000', '002__000', '015__000', '010__002']
Finished retrieval. Saved to: ./output/Customer_Service_Interactions_retrieved_local_30s.json


## Run Inference

In [1]:
import os

# Define base cache directory (modify as needed)
BASE_CACHE_DIR = "/projects/aixpert/users/aravind/cache"

# Uncomment and set specific cache directories as needed
os.environ["TORCH_EXTENSIONS_DIR"] = f"{BASE_CACHE_DIR}/torch"

# Hugging Face cache directories
HF_CACHE_DIR = "{BASE_DIR}/huggingface"
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE_DIR
os.environ["HF_DATASETS_CACHE"] = HF_CACHE_DIR

print("Environment variables set.")

Environment variables set.


In [2]:
import os
import json
import time
import warnings
import torch

from src.model.QwenOmni import Qwen2_5OMNI
from src.system_utils import print_gpu_memory
from src.inference import (
    process_question, # To process a single question and generate an answer using the Qwen2_5OMNI model
)

warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

/fs01/projects/aixpert/users/aravind/ref-imp-5/av-qa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
SYSTEM_PROMPT = """You are a multimodal analyst.
You are given synchronized video and audio content.

Follow this EXACT procedure:

Step 1 — Relevance Check:
If the question CANNOT be answered using the provided video and audio content, respond with EXACTLY:
Unanswerable
Do not add any explanation.

Step 2 — Answering:
If the question IS answerable:

• Provide a concise, factual answer.
• Base the answer strictly on the visible and audible content.
• Include supporting timestamps.
• Format timestamps EXACTLY as: [HH:MM:SS-HH:MM:SS]
• Use only timestamps that appear in the provided segment.
• Do NOT fabricate timestamps.
• Do NOT include reasoning steps.
• Do NOT mention that you analyzed video or audio.

Output Format:

Answer: <concise answer>
Evidence: [HH:MM:SS-HH:MM:SS]
"""

In [6]:
def run_qwen_omni(
        model_name,
        retrieve_path,
        root_data_dir,
        segment_suffix,
        output_path,
        bsz,
        topic,
        max_questions: int = None,
    ):
        """
        Runs the Qwen Omni pipeline to process questions and generate answers.
    
        Args:
            model_name (str): Name of the model to use.
            retrieve_path (str): Path to the file containing questions and retrieved files.
            video_vocabs (str): Path to the directory containing video files.
            audio_vocabs (str): Path to the directory containing audio files.
            output_path (str): Path to save the output results.
            bsz (int): Batch size for processing.
            max_questions (int, optional): Maximum number of questions to process.
    
        Returns:
            None
        """
        total_start = time.time()

        # Initialize Qwen only
        model = Qwen2_5OMNI(model_name=model_name, prompt=SYSTEM_PROMPT)
    
        # Load retrieved results
        with open(retrieve_path, "r") as f:
            sources = json.load(f)
    
        if max_questions is not None:
            sources = sources[:max_questions]
    
        targets = []
        question_times = []
    
        for source in sources:
    
            question_start = time.time()
    
            processed_source = process_question(
                source,
                root_data_dir,
                segment_suffix,
                model,
                bsz,
                topic
            )
    
            targets.append(processed_source)
    
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            with open(output_path, "w") as f:
                json.dump(targets, f, indent=2)
    
            question_duration = time.time() - question_start
            question_times.append(question_duration)
    
            print(f"[Question Completed] {question_duration:.2f}s")
            print("-" * 50)
    
        total_time = time.time() - total_start
        avg_time = sum(question_times) / len(question_times) if question_times else 0
    
        with open(output_path, "w") as f:
            json.dump(targets, f, indent=2)
    
        print("\n" + "="*60)
        print("BOOTCAMP DEMO SUMMARY")
        print("="*60)
        print(f"Total Questions: {len(question_times)}")
        print(f"Average Time / Question: {avg_time:.2f}s")
        print(f"Total Runtime: {total_time:.2f}s")
    
        if torch.cuda.is_available():
            print_gpu_memory()
    
        print("="*60)

### Run Inference on Global Embeddings

In [ ]:
# ===== CONFIGURATION =====

# Model Configuration
MODEL_NAME = "Qwen/Qwen2.5-Omni-7B"  # Name of the model to use

# Topic Configuration
topic = "Customer_Service_Interactions"  # "Patient-Doctor_Consultations" # "Job_Interviews" # "Customer_Service_Interactions"


# Ratio and Term Configuration
suffix = "30s"  # Ratio for processing (e.g., segment length, duration, etc.)
term = "global"  # Retrieval term (e.g., "newRAG", "full")

# File Paths for Input and Output
RETRIEVE_PATH = f"./output/{topic}_retrieved_{term}_{suffix}.json"  # Path to the retrieval file
OUTPUT_PATH = f"./output/{topic}_response_{term}_{suffix}_temp.json"  # Path to save the output

# Vocabulary Directories
# VIDEO_VOCABS = f"./data/{topic}/segment-video_{suffix}"  # Directory for segmented video files
# AUDIO_VOCABS = f"./data/{topic}/segment-audio_{suffix}"  # Directory for segmented audio files
ROOT_DATA_DIR = "./data"

# Batch Size for Processing
BATCH_SIZE = 8  # Number of items to process in a single batch

# Cache Directory
CACHE_DIR = "<path_to_cache_dir>"  # Path to the cache directory (update as needed)

# Maximum questions for processing
MAX_QUESTIONS = 8

# Run the Qwen Omni pipeline for processing questions and generating answers
run_qwen_omni(
    model_name=MODEL_NAME,
    retrieve_path=RETRIEVE_PATH,
    root_data_dir=ROOT_DATA_DIR,
    segment_suffix=suffix,
    output_path=OUTPUT_PATH,
    bsz=BATCH_SIZE,
    topic=topic,
    # max_questions=MAX_QUESTIONS,
)

### Run Inference on local embeddings

In [ ]:
# ===== CONFIGURATION =====

# Model Configuration
MODEL_NAME = "Qwen/Qwen2.5-Omni-7B"  # Name of the model to use
# Batch Size for Processing
BATCH_SIZE = 8  # Number of items to process in a single batch
# Cache Directory
CACHE_DIR = "<path_to_cache_dir>"  # Path to the cache directory (update as needed)



# Topic Configuration
topic = "Customer_Service_Interactions"  # "Patient-Doctor_Consultations" # "Job_Interviews" # "Customer_Service_Interactions"
# Ratio and Term Configuration
suffix = "30s"  # Ratio for processing (e.g., segment length, duration, etc.)
term = "local"  # Retrieval term (e.g., "newRAG", "full", "global", "local")

# File Paths for Input and Output
RETRIEVE_PATH = f"./output/{topic}_retrieved_{term}_{suffix}.json"  # Path to the retrieval file
OUTPUT_PATH = f"./output/{topic}_response_{term}_{suffix}.json"  # Path to save the output

# Vocabulary Directories
# VIDEO_VOCABS = f"./data/{topic}/segment-video_{suffix}"  # Directory for segmented video files
# AUDIO_VOCABS = f"./data/{topic}/segment-audio_{suffix}"  # Directory for segmented audio files
ROOT_DATA_DIR = "./data"

# Maximum questions for processing
MAX_QUESTIONS = 8

# Run the Qwen Omni pipeline for processing questions and generating answers
run_qwen_omni(
    model_name=MODEL_NAME,
    retrieve_path=RETRIEVE_PATH,
    root_data_dir=ROOT_DATA_DIR,
    segment_suffix=suffix,
    output_path=OUTPUT_PATH,
    bsz=BATCH_SIZE,
    topic=topic,
    # max_questions=MAX_QUESTIONS,
)

### References and Resources

1. **[MAGNET: A Multi-agent Framework for Finding Audio-Visual Needles by Reasoning over Multi-Video Haystacks](https://arxiv.org/abs/2506.07016)**  
    A comprehensive framework for temporal, causal, and multi-hop retrieval across long video haystacks.

2. **[VisRAG: Vision-based Retrieval-Augmented Generation on Multi-modality Documents](https://arxiv.org/abs/2410.10594)**  
    A study on retrieval-augmented generation leveraging vision-based multi-modal documents.

3. **[Videos Dataset for LLMs RAG](https://huggingface.co/datasets/elmoghany/Videos-Dataset-For-LLMs-RAG-That-Require-Audio-Vidoes-And-Text)**  
    A dataset designed for retrieval-augmented generation tasks requiring audio, video, and text.

4. **[SONIC-O1 Dataset](https://huggingface.co/datasets/vector-institute/sonic-o1)**  
    A dataset for audio-visual question answering tasks across various domains.